### Exercise 3.2: Fine-tuning a CLIP Model (harder)

Use a (small) CLIP model like [`openai/clip-vit-base-patch16`](https://huggingface.co/openai/clip-vit-base-patch16) and evaluate its zero-shot performance on a small image classification dataset like ImageNette or TinyImageNet. Fine-tune (using a parameter-efficient method!) the CLIP model to see how much improvement you can squeeze out of it.

**Note**: There are several ways to adapt the CLIP model; you could fine-tune the image encoder, the text encoder, or both. Or, you could experiment with prompt learning.

**Tip**: CLIP probably already works very well on ImageNet and ImageNet-like images. For extra fun, look for an image classification dataset with different image types (e.g. *sketches*).

In [1]:
!pip install "datasets<4.0.0" -q


In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # errori CUDA sincroni e leggibili

Setup dell'Ambiente e Caricamento Modello

In [3]:
import torch
import torchvision
import torch.nn.functional as F 
from transformers import CLIPProcessor, CLIPModel, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import os
torch._dynamo.config.disable = True 
torch.backends.cudnn.benchmark = True


device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "openai/clip-vit-base-patch16"
model = CLIPModel.from_pretrained(model_id)
processor = CLIPProcessor.from_pretrained(model_id)
model = model.to(device)
for name, buf in model.named_buffers():
    buf.data = buf.data.to(device)
model.logit_scale.data.fill_(2.6592)
print(f"✅ Modello pronto su {device}. logit_scale resettata a {model.logit_scale.exp().item():.2f}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modello pronto su cuda. logit_scale resettata a 14.28


In [4]:
# Abilita Tensor Cores per matmul più veloci
torch.set_float32_matmul_precision('high')
# Quanti CPU core hai disponibili
print(f"CPU cores disponibili: {os.cpu_count()}")

CPU cores disponibili: 12


Applicazione LoRA

In [5]:
# ✅ NUOVO BLOCCO — LoRA su vision + text encoder, q/k/v/o
lora_config = LoraConfig(
    r=8,               # rank più basso perché ora adattiamo più moduli
    lora_alpha=16,     # mantieni ratio alpha/r = 2
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
)

# Applica a entrambi gli encoder
model.vision_model = get_peft_model(model.vision_model, lora_config)
model.text_model   = get_peft_model(model.text_model,   lora_config)

model.vision_model.print_trainable_parameters()
model.text_model.print_trainable_parameters()

trainable params: 589,824 || all params: 86,389,248 || trainable%: 0.6828
trainable params: 393,216 || all params: 63,559,168 || trainable%: 0.6187


DATASET (completo)

In [6]:
import json, urllib.request

# Label ImageNet standard
url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
urllib.request.urlretrieve(url, "imagenet_labels.json")
with open("imagenet_labels.json") as f:
    imagenet_class_names = json.load(f)

# Caricamento e split
print("Caricamento ImageNet-Sketch...")
full_sketch_ds = load_dataset("imagenet_sketch", split="train", trust_remote_code=True)
full_shuffled = full_sketch_ds.shuffle(seed=42)
train_size = int(0.8 * len(full_shuffled))
sketch_train_ds = full_shuffled.select(range(train_size))
sketch_val_ds   = full_shuffled.select(range(train_size, len(full_shuffled)))
print(f"Train: {len(sketch_train_ds)} | Val: {len(sketch_val_ds)}")

Caricamento ImageNet-Sketch...
Train: 40711 | Val: 10178


In [7]:
# Dataset completo, shuffle + split
full_shuffled = full_sketch_ds.shuffle(seed=42)
train_size = int(0.8 * len(full_shuffled))

sketch_train_ds = full_shuffled.select(range(train_size))
sketch_val_ds   = full_shuffled.select(range(train_size, len(full_shuffled)))
print(f"Train: {len(sketch_train_ds)} | Val: {len(sketch_val_ds)}")

Train: 40711 | Val: 10178


Pre-elaborazione in RAM

In [8]:
# prepare_ram_dataset aggiornata per 1000 classi
def prepare_ram_dataset(dataset, num_samples=None):
    n = num_samples or len(dataset)
    all_pv, all_ids, all_am, all_labels = [], [], [], []
    print(f"Pre-elaborazione di {n} campioni...")
    
    for i in tqdm(range(n)):
        example = dataset[i]
        image = example['image'].convert("RGB")
        local_label = example['label']  # 0-999 direttamente
        prompt = f"a sketch of a {imagenet_class_names[local_label]}"
        
        inputs = processor(
            text=prompt, images=image,
            return_tensors="pt", padding="max_length", truncation=True
        )
        all_pv.append(inputs["pixel_values"])
        all_ids.append(inputs["input_ids"])
        all_am.append(inputs["attention_mask"])
        all_labels.append(local_label)
    
    return TensorDataset(
        torch.cat(all_pv),
        torch.cat(all_ids),
        torch.cat(all_am),
        torch.tensor(all_labels)
    )

# Usa un subset ragionevole per non aspettare ore
train_ram_ds = prepare_ram_dataset(sketch_train_ds, num_samples=5000)
train_loader = DataLoader(
    train_ram_ds, 
    batch_size=64, # Se hai abbastanza VRAM, prova 128 per massimizzare il throughput [7]
    shuffle=True,
    num_workers=0, # Cruciale su Windows per evitare il collo di bottiglia del lock [8]
    pin_memory=True
)
print(f"Batch per epoca: {len(train_loader)}")

Pre-elaborazione di 5000 campioni...


100%|██████████| 5000/5000 [01:53<00:00, 44.06it/s]


Batch per epoca: 79


Valutazione

In [9]:
def evaluate_imagenet_sketch(loader):
    model.eval()
    correct, total = 0, 0
    
    # ✅ Usa le 1000 classi caricate dal JSON
    prompts = [f"a sketch of a {cls}" for cls in imagenet_class_names]
    text_inputs = processor(text=prompts, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        # Estrazione feature di testo per tutte le 1000 classi
        t_outputs = model.text_model(**text_inputs)
        text_features = model.text_projection(t_outputs.pooler_output)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        for batch in tqdm(loader, desc="Valutazione"):
            # Unpack dei 4 elementi (pv, ids, am, labels)
            pv, _, _, labels = batch 
            pv, labels = pv.to(device), labels.to(device)

            # Feature visive "Safe"
            v_outputs = model.vision_model(pixel_values=pv)
            image_features = model.visual_projection(v_outputs.pooler_output)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

            # Calcolo logits contro i 1000 prompt
            logits = (image_features @ text_features.T) * model.logit_scale.exp()
            
            # Predizione corretta se l'indice (0-999) coincide
            preds = logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

Training Loop Contrastivo Ottimizzato

In [10]:
# ==========================================
# 1. CONFIGURAZIONE OTTIMIZZATORE (Specifiche CLIP ViT)
# ==========================================
# ✅ Sostituisci l'optimizer con questo
optimizer = torch.optim.AdamW([
    {"params": filter(lambda p: p.requires_grad, model.vision_model.parameters()),
     "lr": 1e-5},
    {"params": filter(lambda p: p.requires_grad, model.text_model.parameters()),
     "lr": 5e-6},   # più conservativo sul text encoder
    {"params": [model.logit_scale], "lr": 1e-5},
],
    betas=(0.9, 0.98), eps=1e-6, weight_decay=0.2
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=30,
    num_training_steps=len(train_loader) * 5
)

scaler = GradScaler('cuda')
num_epochs = 3

In [11]:
# ==========================================
# 2. OTTIMIZZAZIONE MEMORIA (Fix Rallentamento Windows)
# ==========================================
# Abilitiamo il Gradient Checkpointing: OpenAI lo usa per addestrare CLIP in modo efficiente [2, 3].
# Riduce l'occupazione della VRAM impedendo a Windows di spostare dati nella RAM di sistema (Shared Memory), 
# che è la causa principale del passaggio da <1s a 20s per iterazione [4].
model.vision_model.gradient_checkpointing_enable()
model.text_model.gradient_checkpointing_enable()

CLIPTextTransformer does not expose input embeddings. Gradients cannot flow back to the token embeddings when using adapters or gradient checkpointing. Override `get_input_embeddings` to fully support those features, or set `_input_embed_layer` to the attribute name that holds the embeddings.


In [12]:
# DIAGNOSTICA: verifica range dei label nel train set
pv, ids, am, labels = next(iter(train_loader))
print(f"Label min: {labels.min().item()}, max: {labels.max().item()}")
print(f"Batch size: {pv.shape[0]}")
print(f"Logits shape attesa: [{pv.shape[0]} x {pv.shape[0]}]")
print(f"Cross entropy si aspetta label in [0, {pv.shape[0]-1}]")

Label min: 20, max: 971
Batch size: 64
Logits shape attesa: [64 x 64]
Cross entropy si aspetta label in [0, 63]


In [14]:
# ==========================================
# NUOVO TRAINING LOOP — testo non più frozen
# ==========================================
model.train()
print(f"Inizio training su {device}...")

# Prepara gli input testuali per le 1000 classi UNA VOLTA SOLA (CPU)
all_prompts = [f"a sketch of a {imagenet_class_names[i]}" for i in range(1000)]
# Li tokenizziamo tutti, poi nel loop selezioniamo il batch
all_text_inputs = processor(
    text=all_prompts, return_tensors="pt",
    padding=True, truncation=True
)
# Tieni i tensori tokenizzati su CPU, li slicerai nel loop
all_input_ids      = all_text_inputs["input_ids"]       # [1000, seq_len]
all_attention_mask = all_text_inputs["attention_mask"]  # [1000, seq_len]

for epoch in range(num_epochs):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch in pbar:
        pv, _, _, labels = [v.to(device, non_blocking=True) for v in batch]
        optimizer.zero_grad(set_to_none=True)

        with autocast('cuda'):
            # --- Vision forward ---
            v_out = model.vision_model(pixel_values=pv)
            image_features = model.visual_projection(v_out.pooler_output)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

            # --- Text forward (solo le classi del batch) ---
            labels_cpu = labels.cpu()  # ← indici devono essere su CPU per indexare tensori CPU
            batch_input_ids = all_input_ids[labels_cpu].to(device)
            batch_attn_mask = all_attention_mask[labels_cpu].to(device)
            t_out = model.text_model(
                input_ids=batch_input_ids,
                attention_mask=batch_attn_mask
            )
            text_features = model.text_projection(t_out.pooler_output)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

            # --- Loss contrastiva simmetrica (N×N sul batch) ---
            logits = (image_features @ text_features.T) * model.logit_scale.exp()
            ground_truth = torch.arange(len(logits), device=device)
            loss = (F.cross_entropy(logits, ground_truth) +
                    F.cross_entropy(logits.T, ground_truth)) / 2

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        # Clamp logit_scale per stabilità (CLIP originale clampava a ln(100))
        with torch.no_grad():
            model.logit_scale.clamp_(0, 4.6052)

        current_lr = scheduler.get_last_lr()[0]
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{current_lr:.2e}"})

Inizio training su cuda...


Epoch 3/3: 100%|██████████| 79/79 [03:50<00:00,  2.92s/it, loss=nan, lr=3.95e-06]


In [15]:
from torchvision import transforms

# ── Costanti ──────────────────────────────────────────────────────────────────
IMAGENETTE_INDICES = [0, 217, 482, 491, 497, 566, 569, 571, 574, 701]
imagenette_names   = [imagenet_class_names[i] for i in IMAGENETTE_INDICES]
IMAGENET_IDX_TO_LOCAL = {idx: i for i, idx in enumerate(IMAGENETTE_INDICES)}

# ── Transform CLIP-standard ───────────────────────────────────────────────────
clip_val_transform = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275,  0.40821073],
        std= [0.26862954, 0.26130258, 0.27577711]
    )
])

# ── Collate: filtra solo le 10 classi ────────────────────────────────────────
def val_collate_fn(batch):
    images, labels = [], []
    for item in batch:
        if item['label'] in IMAGENET_IDX_TO_LOCAL:
            images.append(clip_val_transform(item['image'].convert("RGB")))
            labels.append(IMAGENET_IDX_TO_LOCAL[item['label']])  # rimappa 0-9
    if not images:
        return torch.zeros(0), torch.zeros(0, dtype=torch.long)
    return torch.stack(images), torch.tensor(labels)

sketch_val_loader = DataLoader(
    sketch_val_ds,
    batch_size=32,
    collate_fn=val_collate_fn,
    num_workers=0  # collate custom con HF dataset → 0 per sicurezza
)

# ── Funzione di valutazione sulle 10 classi ──────────────────────────────────
def evaluate_imagenette(loader):
    model.eval()
    correct, total = 0, 0

    # Prompt solo per le 10 classi ImageNette
    prompts = [f"a sketch of a {name}" for name in imagenette_names]
    text_inputs = processor(
        text=prompts, return_tensors="pt", padding=True
    ).to(device)

    with torch.no_grad():
        t_out = model.text_model(**text_inputs)
        text_feats = model.text_projection(t_out.pooler_output)
        text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)  # [10, 512]

        for pv, labels in loader:
            if pv.shape[0] == 0:   # batch vuoto (nessuna delle 10 classi)
                continue
            pv     = pv.to(device)
            labels = labels.to(device)

            v_out = model.vision_model(pixel_values=pv)
            img_feats = model.visual_projection(v_out.pooler_output)
            img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)

            logits = (img_feats @ text_feats.T) * model.logit_scale.exp()
            preds  = logits.argmax(dim=-1)   # indici 0-9
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    return correct / total if total > 0 else 0.0

# ── Diagnostica ───────────────────────────────────────────────────────────────
count = sum(1 for item in sketch_val_ds if item['label'] in IMAGENET_IDX_TO_LOCAL)
print(f"Campioni ImageNette nel val set: {count}")

# ── Valutazione finale ────────────────────────────────────────────────────────
print("\n--- Inizio Valutazione Finale Domain Shift (Sketch) ---")

model.vision_model.disable_adapter_layers()
model.text_model.disable_adapter_layers()
baseline_acc = evaluate_imagenette(sketch_val_loader)
print(f"✅ Baseline Zero-Shot Accuracy su Sketch: {baseline_acc*100:.2f}%")

model.vision_model.enable_adapter_layers()
model.text_model.enable_adapter_layers() 
final_acc = evaluate_imagenette(sketch_val_loader)
print(f"✅ Post-Fine-Tuning Accuracy (LoRA) su Sketch: {final_acc*100:.2f}%")

delta = (final_acc - baseline_acc) * 100
print(f"\n🚀 Delta Miglioramento: {delta:+.2f} punti percentuali")

Campioni ImageNette nel val set: 105

--- Inizio Valutazione Finale Domain Shift (Sketch) ---
✅ Baseline Zero-Shot Accuracy su Sketch: 9.52%
✅ Post-Fine-Tuning Accuracy (LoRA) su Sketch: 9.52%

🚀 Delta Miglioramento: +0.00 punti percentuali
